# Классификатор новостных заголовков по 11 категориям (RuBERT)

**Что это.** Дообучение `DeepPavlov/rubert-base-cased` на наших данных:
5757 обучающих, 903 валидационных,
903 тестовых строк и **2194 псевдо-размеченных** строк
из неразмеченных пулов (уверенность ≥ 0.85 + маржа ≥ 0.2 + энтропия ≤ 1.5, балансировка классов).

**Почему так:**
- описание входа заметно помогает на коротких заголовках, часть описаний
  случайно отбрасывается, чтобы модель работала и на источниках без описания;
- Focal Loss (γ=2.0) + веса классов — редкие классы (Здоровье, Лайфстайл, Другие)
  иначе «съедаются» крупными;
- три сида: берём лучшую модель по val macro-F1, это устойчивее одного прогона.

**Время:** 25–50 минут на бесплатной T4.

**Что прислать назад:** скачанный `news_classifier_rubert_model.zip` — я поставлю модель в
приложение, пересоберу APK и замерю качество и скорость на телефоне.

In [ ]:
!pip -q install "transformers>=4.40,<5" onnx onnxruntime
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
import base64, collections, csv, gzip, io, json, os, random, time, zipfile
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CATS = ["Политика и геополитика", "Экономика и бизнес", "Технологии и ИТ", "Наука и космос", "Здоровье и медицина", "Лайфстайл и быт", "Спорт и киберспорт", "Культура и шоу-бизнес", "Происшествия и криминал", "Общество и социальная сфера", "Другие"]
LABEL = {c: i for i, c in enumerate(CATS)}
MAXLEN, BATCH, ACCUM, EPOCHS, LR, DESC_DROP = 96, 32, 1, 5, 2e-5, 0.3
MODEL = 'DeepPavlov/rubert-base-cased'
SEEDS = [42, 7, 2026]
USE_PSEUDO = True

PACKED = "..."
rows = list(csv.DictReader(io.StringIO(gzip.decompress(base64.b64decode(PACKED)).decode('utf-8'))))
splits = {s: [r for r in rows if r['split'] == s and r['category'] in LABEL] for s in ('train', 'val', 'test', 'pseudo')}
# === Псевдо-разметка с фильтрацией по энтропии/марджину + балансировка классов ===
if USE_PSEUDO and len(splits['pseudo']) > 0:
    # Загрузка модели для фильтрации (на CPU)
    from transformers import AutoModelForSequenceClassification
    model_for_filter = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(CATS), token=False).cpu()
    model_for_filter.eval()
    
    import torch
    import collections
    import random
    
    # Подготовка текстов псевдо-меток
    pseudo_texts = []
    for r in splits['pseudo']:
        t, d = r['title'].strip(), (r['description'] or '').strip()
        pseudo_texts.append(t if (not d) else f'{t}. {d}')
    
    # Подсчет вероятностей
    tok = AutoTokenizer.from_pretrained(MODEL, token=False)
    enc = tok(pseudo_texts, truncation=True, max_length=MAXLEN, padding=True, return_tensors='pt')
    with torch.no_grad():
        logits = model_for_filter(**enc).logits
    probs = torch.softmax(logits, dim=-1)
    max_probs, preds = probs.max(dim=-1)
    _, second_probs = torch.topk(probs, 2, dim=-1)
    margins = max_probs - second_probs[:, 0]
    entropies = -(probs * (torch.log(probs + 1e-10))).sum(dim=-1)
    
    # Фильтр: уверенность >= 0.85 И маржа >= 0.2 И энтропия <= 1.5
    confidence = max_probs >= 0.85
    margin_cond = margins >= 0.2
    entropy_cond = entropies <= 1.5
    filter_mask = confidence & margin_cond & entropy_cond
    
    # Оставляем только прошедшие фильтр
    filtered_pseudo_rows = [splits['pseudo'][i] for i in range(len(splits['pseudo'])) if filter_mask[i]]
    print(f'Псевдо-меток после фильтрации: {len(filtered_pseudo_rows)}/{len(splits["pseudo"])}')
    
    # Балансировка классов
    if len(filtered_pseudo_rows) > 0:
        # Группируем по классам
        pseudo_labels = [LABEL[r['category']] for r in filtered_pseudo_rows]
        class_counts = collections.Counter(pseudo_labels)
        min_class_count = min(class_counts.values()) if class_counts else 0
        max_class_count = max(class_counts.values()) if class_counts else 0
        # Переособираем: оставляем минимум классов * коэффициент
        # Для редких классов немного подтягиваем, для частых немного обрезаем
        target_per_class = max(1, int(min_class_count * 1.3))
        
        class_groups = {}
        for r in filtered_pseudo_rows:
            cat_idx = LABEL[r['category']]
            if cat_idx not in class_groups:
                class_groups[cat_idx] = []
            class_groups[cat_idx].append(r)
        
        balanced_rows = []
        for cat_idx in range(len(CATS)):
            items = class_groups.get(cat_idx, [])
            # Перемешиваем
            import random
            random.shuffle(items)
            # Оставляем target_per_class (или меньше если данных мало)
            keep = min(len(items), target_per_class)
            for r in items[:keep]:
                balanced_rows.append(r)
        
        splits['pseudo'] = balanced_rows
        print(f'Псевдо-меток после балансировки: {len(balanced_rows)} (целевых на класс: {target_per_class})')
    else:
        print('Внимание: нет псевдо-меток после фильтрации')
else:
    print('USE_PSEUDO=False, пропуск фильтрации псевдо-меток')

# Пересчитываем train_rows после фильтрации
train_rows = splits['train'] + (splits['pseudo'] if USE_PSEUDO else [])
print('данные:', {k: len(v) for k, v in splits.items()}, '| в обучении:', len(train_rows))
print('распределение обучения:', collections.Counter(r['category'] for r in train_rows).most_common())print('данные:', {k: len(v) for k, v in splits.items()}, '| в обучении:', len(train_rows))
print('распределение обучения:', collections.Counter(r['category'] for r in train_rows).most_common())

In [ ]:
# Модель публичная: ни аккаунт, ни токен Hugging Face не нужны.
# token=False явно говорит библиотеке не искать секреты и не просить вход.
tok = AutoTokenizer.from_pretrained(MODEL, token=False)

class DS(Dataset):
    def __init__(self, rows, train=False, rng=None):
        self.labels = [LABEL[r['category']] for r in rows]
        self.texts = []
        for r in rows:
            t, d = r['title'].strip(), (r['description'] or '').strip()
            drop = train and rng.random() < DESC_DROP
            self.texts.append(t if (drop or not d) else f'{t}. {d}')
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.texts[i], self.labels[i]

def collate(batch):
    texts, labels = zip(*batch)
    enc = tok(list(texts), truncation=True, max_length=MAXLEN, padding=True, return_tensors='pt')
    return {k: v.to(DEVICE) for k, v in enc.items()}, torch.tensor(labels).to(DEVICE)

def macro_f1(y_true, y_pred, n=len(CATS)):
    f1s, precs, recs = [], [], []
    for c in range(n):
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == c and p == c)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != c and p == c)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == c and p != c)
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return sum(f1s) / n, f1s

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    for enc, y in loader:
        logits = model(**enc).logits
        preds.extend(logits.argmax(-1).cpu().tolist())
        trues.extend(y.cpu().tolist())
    acc = sum(p == t for p, t in zip(preds, trues)) / len(trues)
    f1, _ = macro_f1(trues, preds)
    return acc, f1, preds, trues

In [ ]:
def train_one(seed, train_rows, val_rows, epochs=EPOCHS, verbose=True):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    rng = np.random.default_rng(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=len(CATS), token=False).to(DEVICE)
    tr = DataLoader(DS(train_rows, train=True, rng=rng), batch_size=BATCH, shuffle=True, collate_fn=collate)
    va = DataLoader(DS(val_rows), batch_size=64, collate_fn=collate)

    cnt = collections.Counter(LABEL[r['category']] for r in train_rows)
    w = torch.tensor([len(train_rows) / (len(CATS) * max(1, cnt[i])) for i in range(len(CATS))], dtype=torch.float).to(DEVICE)
    w = w / w.mean()
    class FocalLoss(nn.Module):
        def __init__(self, alpha=1.0, gamma=2.0, reduction="mean"):
            super().__init__()
            self.alpha = alpha
            self.gamma = gamma
            self.reduction = reduction
    
        def forward(self, inputs, targets):
            ce_loss = nn.CrossEntropyLoss(reduction="none")(inputs, targets)
            pt = torch.exp(-ce_loss)
            focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
            if self.reduction == "mean":
                return focal_loss.mean()
            elif self.reduction == "sum":
                return focal_loss.sum()
            return focal_loss
    
    lossf = FocalLoss(alpha=w, gamma=2.0)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = max(1, len(tr) // ACCUM) * epochs
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.1)

    best = {'f1': -1, 'state': None, 'epoch': 0, 'acc': 0}
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); run = 0.0; opt.zero_grad()
        for i, (enc, y) in enumerate(tr):
            out = model(**enc)
            loss = lossf(out.logits, y) / ACCUM
            loss.backward(); run += loss.item() * ACCUM
            if (i + 1) % ACCUM == 0 or i + 1 == len(tr):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); sched.step(); opt.zero_grad()
            if verbose and i % 100 == 0:
                print(f'  сид {seed} эпоха {ep} шаг {i}/{len(tr)} loss {run/(i+1):.4f} ({time.time()-t0:.0f}s)', flush=True)
        acc, f1, _, _ = evaluate(model, va)
        print(f'сид {seed} эпоха {ep}: val acc {acc:.4f} | val macro-F1 {f1:.4f} | {time.time()-t0:.0f}s', flush=True)
        if f1 > best['f1']:
            best = {'f1': f1, 'acc': acc, 'epoch': ep,
                    'state': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}}
    model.load_state_dict(best['state'])
    print(f'сид {seed}: лучшая эпоха {best["epoch"]}, val acc {best["acc"]:.4f}, val macro-F1 {best["f1"]:.4f}', flush=True)
    return model, best

results = {}
for seed in SEEDS:
    print(f'\n=== сид {seed} ===', flush=True)
    model, best = train_one(seed, train_rows, splits['val'])
    results[seed] = (model, best)
best_seed = max(results, key=lambda s: results[s][1]['f1'])
model, best = results[best_seed]
print(f'\nЛУЧШИЙ СИД: {best_seed} (val macro-F1 {best["f1"]:.4f})')

In [ ]:
# Честный замер на отложенном тесте (его модель не видела)
te = DataLoader(DS(splits['test']), batch_size=64, collate_fn=collate)
acc, f1, preds, trues = evaluate(model, te)
per = macro_f1(trues, preds)[1]
print(f'ТЕСТ: acc {acc:.4f} | macro-F1 {f1:.4f} | строк {len(trues)}')
for c, f in zip(CATS, per):
    n = sum(1 for t in trues if t == LABEL[c])
    cor = sum(1 for t, p in zip(trues, preds) if t == LABEL[c] and p == LABEL[c])
    print(f'  {c:32s} n={n:3d} recall={cor/max(1,n):.3f} F1={f:.2f}')
conf = collections.Counter((CATS[t], CATS[p]) for t, p in zip(trues, preds) if t != p)
print('\nчастые ошибки:')
for (a, b), k in conf.most_common(12):
    print(f'  {k:3d}  {a[:28]:30s} → {b[:28]}')

## (необязательно) добить качество на train+val

Если тест выше 85% — можно пропустить. Если чуть ниже, запустите эту ячейку:
она переобучает с нуля на `train + val + pseudo` тем же числом эпох и заново
мерит тест. Валидации тут уже нет, поэтому эпохи фиксированы по лучшему сиду.

In [ ]:
USE_TRAINVAL = False   # поставьте True, чтобы включить

if USE_TRAINVAL:
    full = splits['train'] + splits['val'] + (splits['pseudo'] if USE_PSEUDO else [])
    model, best = train_one(best_seed, full, splits['val'], epochs=best['epoch'])
    acc, f1, preds, trues = evaluate(model, te)
    per = macro_f1(trues, preds)[1]
    print(f'ПОСЛЕ ДООБУЧЕНИЯ НА train+val: acc {acc:.4f} | macro-F1 {f1:.4f}')
else:
    print('пропущено')

In [ ]:
# Экспорт для Android: ONNX int8 + vocab.txt + labels.json
import shutil
from pathlib import Path

model = model.to('cpu').eval()
EXP = '/content/export'
shutil.rmtree(EXP, ignore_errors=True); Path(EXP).mkdir(parents=True, exist_ok=True)

vocab = tok.get_vocab()
with open(f'{EXP}/vocab.txt', 'w', encoding='utf-8') as f:
    for token, _id in sorted(vocab.items(), key=lambda kv: kv[1]):
        f.write(token + '\n')
json.dump({'categories': CATS, 'max_length': MAXLEN, 'model': MODEL},
          open(f'{EXP}/labels.json', 'w'), ensure_ascii=False, indent=2)

dummy = tok(['проверка экспорта модели'], return_tensors='pt', max_length=MAXLEN,
            padding='max_length', truncation=True)
# ПОРЯДОК ВАЖЕН: BatchEncoding отдаёт ключи как (input_ids, token_type_ids, attention_mask),
# а forward у BERT ждёт (input_ids, attention_mask, token_type_ids). Если передать
# tuple(dummy.values()), маска и типы токенов меняются местами и модель слепнет.
ORDER = [k for k in ('input_ids', 'attention_mask', 'token_type_ids') if k in dummy]
kwargs = dict(input_names=ORDER, output_names=['logits'],
              dynamic_axes={k: {0: 'batch', 1: 'seq'} for k in ORDER}, opset_version=14)
try:
    torch.onnx.export(model, tuple(dummy[k] for k in ORDER), f'{EXP}/model_fp32.onnx', dynamo=False, **kwargs)
except TypeError:
    torch.onnx.export(model, tuple(dummy[k] for k in ORDER), f'{EXP}/model_fp32.onnx', **kwargs)

from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic(f'{EXP}/model_fp32.onnx', f'{EXP}/model_int8.onnx', weight_type=QuantType.QInt8)
size_mb = os.path.getsize(f'{EXP}/model_int8.onnx') / 1024 / 1024
print(f'model_int8.onnx: {size_mb:.0f} МБ')
os.remove(f'{EXP}/model_fp32.onnx')

In [ ]:
# Проверка: int8 обязан повторять решения PyTorch (хотя бы на 80 тестовых строках)
import onnxruntime as ort
sess = ort.InferenceSession('/content/export/model_int8.onnx', providers=['CPUExecutionProvider'])
INAMES = [i.name for i in sess.get_inputs()]
print('входы ONNX:', INAMES)
model.eval()

def torch_pred(text):
    enc = tok([text], return_tensors='pt', max_length=MAXLEN, padding='max_length', truncation=True)
    with torch.no_grad():
        return int(model(**enc).logits.argmax(-1))

def qpred(text):
    enc = tok([text], max_length=MAXLEN, padding='max_length', truncation=True)
    feeds = {'input_ids': np.array([enc['input_ids']], dtype=np.int64),
             'attention_mask': np.array([enc['attention_mask']], dtype=np.int64),
             'token_type_ids': np.array([enc['token_type_ids']], dtype=np.int64)}
    return int(sess.run(None, {k: v for k, v in feeds.items() if k in INAMES})[0][0].argmax())

sample = splits['test'][:80]
same = 0
for r in sample:
    text = f"{r['title']}. {r['description']}" if r['description'] else r['title']
    if torch_pred(text) == qpred(text):
        same += 1
print(f'int8 совпал с PyTorch: {same}/{len(sample)}')
assert same >= len(sample) * 0.95, 'int8 сильно расходится — проверьте порядок входов!'

metrics = {'model': MODEL, 'max_length': MAXLEN, 'best_seed': int(best_seed),
           'train_rows': len(train_rows), 'pseudo': USE_PSEUDO,
           'test_acc': acc, 'test_macro_f1': f1,
           'per_class_f1': per, 'categories': CATS,
           'errors': [[a, b, k] for (a, b), k in conf.most_common(20)]}
json.dump(metrics, open('/content/export/metrics.json', 'w'), ensure_ascii=False, indent=2)
print(json.dumps({k: v for k, v in metrics.items() if k not in ('per_class_f1', 'categories', 'errors')},
                 ensure_ascii=False))

In [ ]:
# Архив для передачи в приложение
zip_path = '/content/news_classifier_rubert_model.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ('model_int8.onnx', 'vocab.txt', 'labels.json', 'metrics.json'):
        z.write(f'/content/export/{name}', name)
print('архив:', zip_path, f'{os.path.getsize(zip_path)/1024/1024:.0f} МБ')
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print('скачайте вручную из панели файлов:', e)

## Готово

Модель обучена и выгружена для инференса на устройстве: `model_int8.onnx`, `vocab.txt`, `labels.json`, `metrics.json`.

Если качество на тесте ниже ожидаемого, следующий шаг — предобучение на дополнительном корпусе новостей.